# DermaLens — Synthetic facial image prompt-family split (Kaggle)

**Goal:** split six classes into `train/`, `val/`, `test/` **without putting the same prompt family in different partitions**. This is for **synthetic images**; it deliberately does **not** use pHash, dHash, ResNet, ORB, or similarity cutoffs. Similar-looking, independently generated images are not assumed to be duplicates.

**Pipeline:** prompt metadata → validate images → SHA-256 & decoded RGB pixel SHA-256 → quarantine identical images with conflicting labels → remove exact copies → keep related prompt families together → 70/15/15 group-aware split → independent leakage checks → ZIP.

1. Attach your synthetic dataset(s) in Kaggle **Add Input**. Update **only the configuration cell** below.
2. Recommended: a CSV with `image_path,class_name,prompt_id,prompt_family_id` (family ID may be blank; it then defaults to prompt ID). The `image_path` can be absolute, or relative to `IMAGE_ROOT`.
3. Alternatively select `INPUT_MODE = "filenames"`: files must have consistent names such as `ACNE_P001_seed101.png`, and you must set all six `CLASS_DIRS`. Optionally supply a prompt-family mapping CSV. Do **not** attempt filename mode if the names do not actually encode prompt IDs.
4. Kaggle: run **Run All**. Download `/kaggle/working/synthetic_facial_split.zip` from **Notebook Output**.

**Important:** For a genuine prompt-family holdout, supply the same `prompt_family_id` to paraphrases or variants of one base prompt. The code cannot infer prompt relatedness from photographs. Exact duplicates with inconsistent classes are quarantined and reported. The approximate split percentages may deviate if groups are uneven; each class must have at least three independent prompt groups. All original image bytes are preserved.


In [ ]:
# ===== EDIT THIS CELL ONLY =====
INPUT_MODE = "manifest"             # "manifest" (recommended) OR "filenames"

# Manifest mode: attach a CSV generated alongside your synthetic image generation.
MANIFEST_CSV = "/kaggle/input/EDIT_SYNTHETIC_DATASET/synthetic_metadata.csv"
IMAGE_ROOT = "/kaggle/input/EDIT_SYNTHETIC_DATASET"  # base for relative image_path entries

# Filename mode: only used when INPUT_MODE == "filenames".
# Pattern must contain named group (?P<prompt_id>...) and reliably encode each prompt ID.
CLASS_DIRS = {
    "Acne": "/kaggle/input/EDIT_SYNTHETIC_DATASET/Acne",
    "Fungal_Infection": "/kaggle/input/EDIT_SYNTHETIC_DATASET/Fungal_Infection",
    "Vitiligo": "/kaggle/input/EDIT_SYNTHETIC_DATASET/Vitiligo",
    "Hyperpigmentation": "/kaggle/input/EDIT_SYNTHETIC_DATASET/Hyperpigmentation",
    "Skin_Cancer": "/kaggle/input/EDIT_SYNTHETIC_DATASET/Skin_Cancer",
    "Normal": "/kaggle/input/EDIT_SYNTHETIC_DATASET/Normal",
}
FILENAME_PROMPT_REGEX = r"^(?P<prompt_id>.+?)_seed[_-]?\d+(?:_.*)?$"
# Optional CSV columns: class_name,prompt_id,prompt_family_id
PROMPT_FAMILY_MAP_CSV = ""

SEED = 42
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
OUTPUT_DIR = "/kaggle/working/dermalens_synthetic_split"
ZIP_PATH = "/kaggle/working/synthetic_facial_split.zip"
VERIFY_ZIP_CRC = True  # may take longer for a very large image dataset

# Supported image extensions (original files are copied; no re-encoding)
EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
CLASS_NAMES = ["Acne", "Fungal_Infection", "Vitiligo", "Hyperpigmentation", "Skin_Cancer", "Normal"]


In [ ]:
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import sys
from collections import Counter, defaultdict
from PIL import Image, ImageOps, __version__ as pillow_version
import zipfile

assert INPUT_MODE in ("manifest", "filenames")
assert set(CLASS_NAMES) == set(CLASS_DIRS) and len(CLASS_NAMES) == 6
assert set(SPLIT_FRACTIONS) == {"train", "val", "test"}
assert all(v > 0 for v in SPLIT_FRACTIONS.values())
assert math.isclose(sum(SPLIT_FRACTIONS.values()), 1.0, abs_tol=1e-9)
assert len(set(CLASS_NAMES)) == 6
print(f"Python: {sys.version.split()[0]}; Pillow: {pillow_version}; input mode: {INPUT_MODE}")


def csv_write(path, rows, columns):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def read_csv(path):
    with open(path, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames:
            raise ValueError(f"No CSV column headers found: {path}")
        return list(reader), set(reader.fieldnames)


def nonblank(value):
    return str(value or "").strip()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def exact_pixel_hash(path):
    # Normalize EXIF orientation; compare decoded RGB pixels, not compressed file bytes.
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        rgb = image.convert("RGB")
        w, h = rgb.size
        if w < 1 or h < 1:
            raise ValueError("zero-size image")
        digest = hashlib.sha256()
        digest.update(f"RGB:{w}x{h}:".encode("ascii"))
        digest.update(rgb.tobytes())
        return digest.hexdigest(), w, h


class UnionFind:
    def __init__(self, items):
        self.parent = {item: item for item in items}

    def find(self, a):
        p = self.parent[a]
        if p != a:
            self.parent[a] = self.find(p)
        return self.parent[a]

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            # Deterministic even if a parent chain has a different shape.
            self.parent[max(a, b)] = min(a, b)


def stable_id(s, length=16):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:length]


In [ ]:
# ===== LOAD INPUT METADATA / FILE INVENTORY =====

def load_records():
    recs = []
    if INPUT_MODE == "manifest":
        manifest_path = Path(MANIFEST_CSV).expanduser()
        if not manifest_path.is_file():
            raise FileNotFoundError(
                f"Manifest not found: {manifest_path}. Edit MANIFEST_CSV in configuration. "
                "Required CSV columns: image_path,class_name,prompt_id; prompt_family_id recommended."
            )
        rows, headers = read_csv(manifest_path)
        must_have = {"image_path", "class_name", "prompt_id"}
        if not must_have.issubset(headers):
            raise ValueError(f"Missing CSV columns: {sorted(must_have - headers)}")
        base = Path(IMAGE_ROOT).expanduser() if nonblank(IMAGE_ROOT) else manifest_path.parent
        for row_num, row in enumerate(rows, start=2):
            label = nonblank(row.get("class_name"))
            path_str = nonblank(row.get("image_path"))
            prompt_id = nonblank(row.get("prompt_id"))
            family_id = nonblank(row.get("prompt_family_id")) or prompt_id
            if label not in CLASS_NAMES:
                raise ValueError(f"CSV row {row_num}: unknown class_name {label!r}; expected {CLASS_NAMES}")
            if not path_str or not prompt_id or not family_id:
                raise ValueError(f"CSV row {row_num}: image_path and prompt_id cannot be blank")
            p = Path(path_str).expanduser()
            if not p.is_absolute():
                p = base / p
            p = p.resolve()
            if not p.is_file():
                raise FileNotFoundError(f"CSV row {row_num}: missing image: {p}")
            if p.suffix.lower() not in EXTENSIONS:
                raise ValueError(f"CSV row {row_num}: unsupported image extension: {p}")
            recs.append({"image_path": str(p), "class_name": label,
                         "prompt_id": prompt_id, "prompt_family_id": family_id,
                         "seed": nonblank(row.get("seed")),
                         "model_id": nonblank(row.get("model_id"))})
    else:
        family_map = {}
        if nonblank(PROMPT_FAMILY_MAP_CSV):
            mapping_rows, headers = read_csv(PROMPT_FAMILY_MAP_CSV)
            if not {"class_name", "prompt_id", "prompt_family_id"}.issubset(headers):
                raise ValueError("Family mapping requires: class_name,prompt_id,prompt_family_id")
            for row in mapping_rows:
                key = (nonblank(row["class_name"]), nonblank(row["prompt_id"]))
                fam = nonblank(row["prompt_family_id"])
                if not all(key) or not fam:
                    raise ValueError("Blank class/prompt/family in family mapping CSV")
                if key in family_map and family_map[key] != fam:
                    raise ValueError(f"Conflicting mapping for {key}")
                family_map[key] = fam
        if "prompt_id" not in re.compile(FILENAME_PROMPT_REGEX).groupindex:
            raise ValueError("FILENAME_PROMPT_REGEX needs the named capture (?P<prompt_id>...)")
        for label in CLASS_NAMES:
            folder = Path(CLASS_DIRS[label]).expanduser()
            if not folder.is_dir():
                raise FileNotFoundError(f"Missing {label} folder: {folder}")
            paths = sorted(p.resolve() for p in folder.rglob("*")
                           if p.is_file() and p.suffix.lower() in EXTENSIONS)
            for p in paths:
                match = re.fullmatch(FILENAME_PROMPT_REGEX, p.stem, flags=re.IGNORECASE)
                if not match or not nonblank(match.group("prompt_id")):
                    raise ValueError(
                        f"Cannot extract prompt_id from {p.name!r}. "
                        "Use a metadata CSV instead, or edit FILENAME_PROMPT_REGEX."
                    )
                prompt_id = nonblank(match.group("prompt_id"))
                family_id = family_map.get((label, prompt_id), prompt_id)
                recs.append({"image_path": str(p), "class_name": label,
                             "prompt_id": prompt_id, "prompt_family_id": family_id,
                             "seed": "", "model_id": ""})
    if not recs:
        raise ValueError("No images found. Check your Kaggle attached input paths.")

    seen_paths = {}
    prompt_to_family = {}
    for rec in recs:
        p = rec["image_path"]
        if p in seen_paths:
            raise ValueError(f"Image listed more than once: {p}. Remove duplicate manifest rows.")
        seen_paths[p] = True
        key = (rec["class_name"], rec["prompt_id"])
        family = rec["prompt_family_id"]
        if key in prompt_to_family and prompt_to_family[key] != family:
            raise ValueError(f"Prompt {key} maps to multiple family IDs; fix the metadata.")
        prompt_to_family[key] = family

    # Keep input order independent of how the metadata CSV was exported.
    return sorted(recs, key=lambda r: (r["class_name"], r["image_path"]))

records = load_records()
print("Images in metadata:", len(records))
print("By class:", dict(Counter(r["class_name"] for r in records)))
print("Prompt families per class:", {c: len({r["prompt_family_id"] for r in records
                                         if r["class_name"] == c}) for c in CLASS_NAMES})


In [ ]:
# ===== EXACT HASHES, INVALID IMAGES, CLASS-CONFLICT QUARANTINE =====
valid = []
invalid_images = []
for i, rec in enumerate(records, 1):
    try:
        pix_hash, width, height = exact_pixel_hash(rec["image_path"])
        file_hash = sha256_file(rec["image_path"])
        valid.append({**rec, "file_sha256": file_hash, "pixel_sha256": pix_hash,
                      "width": width, "height": height})
    except (OSError, ValueError, Image.DecompressionBombError) as exc:
        invalid_images.append({**rec, "error": f"{type(exc).__name__}: {exc}"})
    if i % 500 == 0 or i == len(records):
        print(f"Checked {i}/{len(records)} images")

uf = UnionFind(range(len(valid)))
by_hash = {}
for i, rec in enumerate(valid):
    for hash_name in ("file_sha256", "pixel_sha256"):
        key = (hash_name, rec[hash_name])
        if key in by_hash:
            uf.union(i, by_hash[key])
        else:
            by_hash[key] = i

components = defaultdict(list)
for i in range(len(valid)):
    components[uf.find(i)].append(i)

kept = []
removed_duplicates = []
quarantined_conflicts = []
# Internal prompt-family unions only when *exactly identical* images bridge two families.
family_keys = {(r["class_name"], r["prompt_family_id"]) for r in valid}
family_uf = UnionFind(family_keys)
for members in components.values():
    items = sorted((valid[i] for i in members), key=lambda r: r["image_path"])
    labels = {r["class_name"] for r in items}
    if len(labels) > 1:
        conflict_id = "C_" + stable_id("|".join(sorted(r["image_path"] for r in items)))
        for item in items:
            quarantined_conflicts.append({**item, "conflict_id": conflict_id,
                                          "conflicting_classes": ";".join(sorted(labels))})
        continue
    canonical = items[0]
    kept.append(canonical)
    for extra in items[1:]:
        removed_duplicates.append({**extra, "canonical_path": canonical["image_path"],
                                   "reason": "exact file/pixel duplicate"})
        family_uf.union((canonical["class_name"], canonical["prompt_family_id"]),
                        (extra["class_name"], extra["prompt_family_id"]))

families_by_root = defaultdict(set)
for rec in valid:
    fkey = (rec["class_name"], rec["prompt_family_id"])
    families_by_root[family_uf.find(fkey)].add(rec["prompt_family_id"])
for rec in kept:
    root = family_uf.find((rec["class_name"], rec["prompt_family_id"]))
    family_names = sorted(families_by_root[root])
    rec["group_id"] = "G_" + stable_id(rec["class_name"] + "|" + "|".join(family_names))
    rec["merged_families"] = ";".join(family_names)

print("Valid:", len(valid), "invalid:", len(invalid_images),
      "exact duplicates removed:", len(removed_duplicates),
      "class-conflicting images quarantined:", len(quarantined_conflicts),
      "retained:", len(kept))
if invalid_images:
    print("Warning: corrupt/unsupported images will be excluded and logged.")
if quarantined_conflicts:
    print("Warning: images with conflicting class labels were quarantined; review their report.")


In [ ]:
# ===== PROMPT-GROUP-AWARE, CLASS-BALANCED TRAIN / VAL / TEST SPLIT =====

def assign_one_class_group_splits(class_rows, class_name):
    sizes = Counter(r["group_id"] for r in class_rows)
    if len(sizes) < 3:
        raise ValueError(
            f"{class_name} has only {len(sizes)} independent prompt-family group(s) after "
            "deduplication and conflict quarantine. At least 3 are needed "
            "to place one group in train, val and test. Generate more distinct prompts."
        )
    targets = {s: fraction * sum(sizes.values()) for s, fraction in SPLIT_FRACTIONS.items()}
    split_names = list(SPLIT_FRACTIONS)
    best_solution = None
    best_cost = float("inf")

    # Repeated, deterministic greedy starts improve group-size balance without
    # ever splitting a prompt-family group. No dependency on scikit-learn.
    for trial in range(80):
        rng = random.Random(SEED + trial * 7919 + int(stable_id(class_name), 16))
        groups = list(sizes)
        rng.shuffle(groups)
        groups.sort(key=lambda g: -(sizes[g] * (0.80 + 0.40 * rng.random())))
        counts = {s: 0 for s in split_names}
        groups_in_split = {s: 0 for s in split_names}
        assignments = {}
        for pos, group in enumerate(groups):
            remaining = len(groups) - pos
            missing = [s for s in split_names if groups_in_split[s] == 0]
            choices = missing if len(missing) == remaining else split_names
            trial_choices = []
            for s in choices:
                updated = dict(counts)
                updated[s] += sizes[group]
                score = sum(abs(updated[name] - targets[name])
                            for name in split_names)
                # Tie-break with a seeded shuffle; keeps outcomes deterministic.
                trial_choices.append((score, rng.random(), s))
            chosen = min(trial_choices)[2]
            assignments[group] = chosen
            counts[chosen] += sizes[group]
            groups_in_split[chosen] += 1

        # Improve by moving single groups where a split retains >= 1 group.
        for _ in range(4):
            improved = False
            for group in sorted(groups, key=lambda g: (sizes[g], g)):
                current = assignments[group]
                if groups_in_split[current] <= 1:
                    continue
                base = sum(abs(counts[s] - targets[s]) for s in split_names)
                for destination in split_names:
                    if destination == current:
                        continue
                    new_counts = dict(counts)
                    new_counts[current] -= sizes[group]
                    new_counts[destination] += sizes[group]
                    candidate = sum(abs(new_counts[s] - targets[s]) for s in split_names)
                    if candidate < base - 1e-9:
                        assignments[group] = destination
                        counts = new_counts
                        groups_in_split[current] -= 1
                        groups_in_split[destination] += 1
                        improved = True
                        break
            if not improved:
                break
        cost = sum(abs(counts[s] - targets[s]) for s in split_names)
        if cost < best_cost:
            best_cost = cost
            best_solution = assignments.copy()
    return best_solution

assignments = {}
for label in CLASS_NAMES:
    class_recs = [r for r in kept if r["class_name"] == label]
    choices = assign_one_class_group_splits(class_recs, label)
    for group_id, split_name in choices.items():
        if group_id in assignments and assignments[group_id] != split_name:
            raise AssertionError("Group ID collision across classes")
        assignments[group_id] = split_name
for rec in kept:
    rec["split"] = assignments[rec["group_id"]]

split_statistics = []
for label in CLASS_NAMES:
    subset = [r for r in kept if r["class_name"] == label]
    row = {"class_name": label, "total": len(subset),
           "independent_groups": len({r["group_id"] for r in subset})}
    for s in SPLIT_FRACTIONS:
        n = sum(r["split"] == s for r in subset)
        row[s] = n
        row[s + "_pct"] = round(100 * n / len(subset), 2) if subset else 0
        if n == 0:
            raise AssertionError(f"Empty {label}/{s}; need >= 3 independent prompt groups")
    split_statistics.append(row)
print("Realized split statistics (percentages are targets, not guarantees):")
for row in split_statistics:
    print(row)


In [ ]:
# ===== OUTPUT COPY, REPORTS, LEAKAGE AUDIT, ZIP =====
output = Path(OUTPUT_DIR).expanduser().resolve()
zip_path = Path(ZIP_PATH).expanduser().resolve()
if output == zip_path or output in zip_path.parents:
    raise ValueError("ZIP_PATH must be outside OUTPUT_DIR to avoid recursive archive inclusion")
if output.exists():
    shutil.rmtree(output)
reports = output / "reports"
reports.mkdir(parents=True)
for split_name in SPLIT_FRACTIONS:
    for label in CLASS_NAMES:
        (output / split_name / label).mkdir(parents=True, exist_ok=True)

for rec in kept:
    src = Path(rec["image_path"])
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", src.stem)[:60].strip("._") or "image"
    unique_key = stable_id(str(src), 18)
    fname = f"{cleaned}__{unique_key}{src.suffix.lower()}"
    relpath = Path(rec["split"]) / rec["class_name"] / fname
    dst = output / relpath
    if dst.exists():
        raise RuntimeError(f"Destination collision: {dst}")
    shutil.copy2(src, dst)   # original bytes: no resizing, no recompression
    rec["output_path"] = relpath.as_posix()
    rec["output_sha256"] = sha256_file(dst)
    if rec["output_sha256"] != rec["file_sha256"]:
        raise RuntimeError(f"Copy checksum failed for: {src}")

# Audit *all* known prompt and exact image keys, including family merges.
violations = []
for key_name, key_func in (
    ("group_id", lambda r: r["group_id"]),
    ("prompt_id", lambda r: (r["class_name"], r["prompt_id"])),
    ("prompt_family_id", lambda r: (r["class_name"], r["prompt_family_id"])),
    ("file_sha256", lambda r: r["file_sha256"]),
    ("pixel_sha256", lambda r: r["pixel_sha256"]),
):
    locations = defaultdict(set)
    for rec in kept:
        locations[key_func(rec)].add(rec["split"])
    violations += [{"type": key_name, "key": str(k), "splits": sorted(v)}
                   for k, v in locations.items() if len(v) > 1]
if violations:
    raise AssertionError(f"Leakage audit failed: {violations[:5]}")

copy_count = len(list(output.glob("train/**/*")))  # informational only
manifest_cols = ["output_path", "image_path", "class_name", "prompt_id", "prompt_family_id",
                 "merged_families", "group_id", "split", "seed", "model_id",
                 "file_sha256", "pixel_sha256", "output_sha256", "width", "height"]
source_cols = ["image_path", "class_name", "prompt_id", "prompt_family_id",
               "seed", "model_id", "file_sha256", "pixel_sha256", "width", "height"]
csv_write(reports / "split_manifest.csv", kept, manifest_cols)
csv_write(reports / "removed_exact_duplicates.csv", removed_duplicates,
          source_cols + ["canonical_path", "reason"])
csv_write(reports / "quarantined_label_conflicts.csv", quarantined_conflicts,
          source_cols + ["conflict_id", "conflicting_classes"])
csv_write(reports / "invalid_images.csv", invalid_images,
          ["image_path", "class_name", "prompt_id", "prompt_family_id", "error"])
csv_write(reports / "split_statistics.csv", split_statistics,
          ["class_name", "total", "independent_groups", "train", "train_pct",
           "val", "val_pct", "test", "test_pct"])
group_rows = {}
for r in kept:
    if r["group_id"] not in group_rows:
        group_rows[r["group_id"]] = {"group_id": r["group_id"],
                                     "class_name": r["class_name"],
                                     "merged_families": r["merged_families"],
                                     "split": r["split"], "image_count": 0}
    group_rows[r["group_id"]]["image_count"] += 1
csv_write(reports / "prompt_group_assignments.csv", list(group_rows.values()),
          ["group_id", "class_name", "merged_families", "split", "image_count"])

summary = {
    "input_mode": INPUT_MODE,
    "split_targets": SPLIT_FRACTIONS,
    "seed": SEED,
    "original_images": len(records),
    "valid_images": len(valid),
    "invalid_images": len(invalid_images),
    "removed_exact_duplicates": len(removed_duplicates),
    "quarantined_label_conflicts": len(quarantined_conflicts),
    "final_images": len(kept),
    "classes": split_statistics,
    "known_cross_split_leakage_violations": len(violations),
    "methods": ["prompt-family holdout", "SHA-256", "decoded-RGB-pixel-SHA-256"],
    "limitations": [
        "A family ID is only as reliable as its generation metadata; paraphrases must share a family ID.",
        "No perceptual or embedding duplicate search is performed.",
        "A clean audit cannot rule out visually similar nonidentical samples or generator training-data leakage.",
        "Split fractions can deviate from targets due to indivisible prompt groups.",
    ],
}
(reports / "dataset_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
audit = {"pass": len(violations) == 0,
         "checks": ["prompt ID isolation", "prompt family ID isolation", "merged group isolation",
                    "exact-file-hash isolation", "exact-pixel-hash isolation",
                    "original-byte image copy checksums", "nonempty class split folders"],
         "violations": violations, "images_verified": len(kept),
         "quarantined_label_conflicts": len(quarantined_conflicts)}
(reports / "leakage_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")

zip_path.parent.mkdir(parents=True, exist_ok=True)
tmp_zip = zip_path.with_name(zip_path.name + ".partial")
if tmp_zip.exists():
    tmp_zip.unlink()
try:
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
        for s in SPLIT_FRACTIONS:
            for c in CLASS_NAMES:
                zf.writestr(f"{s}/{c}/", b"")
        for p in sorted(output.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=p.relative_to(output).as_posix())
    with zipfile.ZipFile(tmp_zip, "r", allowZip64=True) as zf:
        names = set(zf.namelist())
        expected_images = {r["output_path"] for r in kept}
        if not expected_images.issubset(names):
            raise AssertionError("Missing copied image(s) from ZIP")
        for s in SPLIT_FRACTIONS:
            for c in CLASS_NAMES:
                if f"{s}/{c}/" not in names:
                    raise AssertionError(f"Missing directory in ZIP: {s}/{c}")
        if VERIFY_ZIP_CRC:
            bad_entry = zf.testzip()
            if bad_entry:
                raise AssertionError(f"ZIP CRC check failed for {bad_entry}")
    os.replace(tmp_zip, zip_path)
except Exception:
    if tmp_zip.exists():
        tmp_zip.unlink()
    raise

print("\nCOMPLETE — known prompt/exact-duplicate leakage checks PASSED")
print("Download this ZIP from Kaggle Output:", zip_path)
print("ZIP size (MiB):", round(zip_path.stat().st_size / (1024 ** 2), 2))
print("Final images:", len(kept), "Prompt groups:", len(group_rows))
print("Quarantined conflicts:", len(quarantined_conflicts),
      "Removed exact duplicates:", len(removed_duplicates), "Invalid:", len(invalid_images))
print("All original image files were copied without modification.")


## Output structure and research usage

```text
synthetic_facial_split.zip
├── train/           # 6 class folders, original images
├── val/             # 6 class folders, original images
├── test/            # 6 class folders, original images
└── reports/
    ├── split_manifest.csv
    ├── prompt_group_assignments.csv
    ├── removed_exact_duplicates.csv
    ├── quarantined_label_conflicts.csv
    ├── invalid_images.csv
    ├── split_statistics.csv
    ├── dataset_summary.json
    └── leakage_audit.json
```

Use the **same frozen split** for all classifier architectures. Classifier hyperparameters may be tuned on `val`, not `test`. For the *real* test set, verify that your diffusion model or LoRA did **not** train on that held-out real test data. Do not interpret a clean prompt-family split as a guarantee that visually similar/generated samples or upstream memorization cannot leak information.

**If a class has fewer than three independent prompt-family groups,** the notebook stops explicitly rather than silently creating an empty test/validation split. Generate more independent prompts or supply corrected family IDs.
